In [ ]:
import torch
import json

# data = torch.load("/data/zhoulingfeng/data_cutLib/save_subgraph/save_designs_b16/iwls2022/blockset_ex27_hop3_batch16.pt")
with open("/data/zhoulingfeng/data_cutLib/cutLibData2/save_json_OpenABC/ac97_ctrl_orig/data_2.json", "r") as f:
    data = json.load(f)

In [ ]:
data['cell_struct'].keys()

In [ ]:
data['cell_struct']['2345']['feats']['feat_names']

In [ ]:
import numpy as np
np.array(data['cell_struct']['2345']['feats']['node_feats'])[:,7]

In [ ]:
def build_fanin_graph_from_feat_graph_dict(feat_graph_raw):
    feat_graph = {
        int(src): [(int(dst), int(is_inv)) for dst, is_inv in fanouts]
        for src, fanouts in feat_graph_raw.items()
    }

    fanin_graph = {node: [] for node in feat_graph}
    for fanouts in feat_graph.values():
        for dst, _ in fanouts:
            if dst not in fanin_graph:
                fanin_graph[dst] = []

    for src, fanouts in feat_graph.items():
        for dst, is_inv in fanouts:
            fanin_graph[dst].append((src, is_inv))

    return fanin_graph


In [ ]:
import torch
import json

dataset1 = "OpenABC"
case1 = "ac97_ctrl_orig"
# data = torch.load("/data/zhoulingfeng/data_cutLib/save_subgraph/save_designs_b16/iwls2022/blockset_ex27_hop3_batch16.pt")
with open(f"/data/zhoulingfeng/data_cutLib/cutLibData2/save_json_{dataset1}/{case1}/data_2.json", "r") as f:
    data_1 = json.load(f)

with open(f"/home/zhoulingfeng/EDAProject/cutLiborigin/data/case_full_graphs/{dataset1}/{case1}/transformed_graph.json", "r") as f:
    graph_1 = json.load(f)
with open(f"/home/zhoulingfeng/EDAProject/cutLiborigin/data/case_full_graphs/{dataset1}/{case1}/feat_graph.json", "r") as f:
    feat_graph_1 = build_fanin_graph_from_feat_graph_dict(json.load(f))

In [ ]:

dataset2 = "forgeEDA"
case2 = "InstructionCache"
# data = torch.load("/data/zhoulingfeng/data_cutLib/save_subgraph/save_designs_b16/iwls2022/blockset_ex27_hop3_batch16.pt")
with open(f"/data/zhoulingfeng/data_cutLib/cutLibData2/save_json_{dataset2}/{case2}/data_2.json", "r") as f:
    data_2 = json.load(f)
with open(f"/home/zhoulingfeng/EDAProject/cutLiborigin/data/case_full_graphs/{dataset2}/{case2}/transformed_graph.json", "r") as f:
    graph_2 = json.load(f)
with open(f"/home/zhoulingfeng/EDAProject/cutLiborigin/data/case_full_graphs/{dataset2}/{case2}/feat_graph.json", "r") as f:
    feat_graph_2 = build_fanin_graph_from_feat_graph_dict(json.load(f))

In [ ]:
import numpy as np
np.array(data_1['cell_struct']['2345']['feats']['feats'][8:]) - np.array(data_2['cell_struct']['39224']['feats']['feats'][8:])

In [ ]:
data_2['cell_struct']['39224']['feats']['subgraph']

In [ ]:
data['cell_struct']['2345']['feats']['subgraph']

In [ ]:
graph_1['4']

In [ ]:
data_1['cell_struct']['2345']['feats']['feats'][:8]

In [ ]:
data_2['cell_struct']['39224']['feats']['feats'][:8]

In [ ]:
data_1['cell_struct']['2345']['abc_delay'], data_2['cell_struct']['39224']['abc_delay']


In [ ]:
feat_graph_2[2342]

In [ ]:
graph_2['17959']

In [ ]:
import json
from collections import deque, defaultdict
import networkx as nx
import matplotlib.pyplot as plt


def normalize_graph(graph_raw):
    return {
        int(src): [(int(dst), int(is_inv)) for dst, is_inv in fanouts]
        for src, fanouts in graph_raw.items()
    }


def collect_nodes(graph):
    nodes = set(graph.keys())
    for fanouts in graph.values():
        for dst, _ in fanouts:
            nodes.add(dst)
    return nodes


def build_fanin_graph(feat_graph):
    fanin_graph = {node: [] for node in collect_nodes(feat_graph)}
    for src, fanouts in feat_graph.items():
        for dst, is_inv in fanouts:
            fanin_graph.setdefault(dst, [])
            fanin_graph[dst].append((src, is_inv))
    return fanin_graph


def multi_source_expand(graph, start_nodes, max_depth):
    visited = {node: 0 for node in start_nodes}
    q = deque((node, 0) for node in start_nodes)

    while q:
        node, depth = q.popleft()
        if depth >= max_depth:
            continue

        for nxt, _ in graph.get(node, []):
            nd = depth + 1
            if nxt not in visited or nd < visited[nxt]:
                visited[nxt] = nd
                q.append((nxt, nd))

    return visited


def build_context_from_all_core_nodes(subgraph_raw, feat_graph_raw, up_n=2, down_n=2):
    subgraph = normalize_graph(subgraph_raw)
    full_feat_graph = normalize_graph(feat_graph_raw)
    full_fanin_graph = build_fanin_graph(full_feat_graph)

    core_nodes = collect_nodes(subgraph)

    up_reach = multi_source_expand(full_fanin_graph, core_nodes, up_n)
    down_reach = multi_source_expand(full_feat_graph, core_nodes, down_n)

    up_nodes = {n for n, d in up_reach.items() if d > 0}
    down_nodes = {n for n, d in down_reach.items() if d > 0}
    all_nodes = set(core_nodes) | up_nodes | down_nodes

    edges = []
    for src in all_nodes:
        for dst, is_inv in full_feat_graph.get(src, []):
            if dst in all_nodes:
                edges.append((src, dst, is_inv))

    node_role = {}
    for node in all_nodes:
        if node in core_nodes:
            node_role[node] = "core"
        elif node in up_nodes and node in down_nodes:
            node_role[node] = "both"
        elif node in up_nodes:
            node_role[node] = "up"
        else:
            node_role[node] = "down"

    return {
        "core_nodes": core_nodes,
        "all_nodes": all_nodes,
        "edges": edges,
        "node_role": node_role,
        "up_dist": up_reach,
        "down_dist": down_reach,
    }


def compute_topological_layers(nodes, edges):
    succ = {n: [] for n in nodes}
    indeg = {n: 0 for n in nodes}

    for src, dst, _ in edges:
        succ[src].append(dst)
        indeg[dst] += 1

    q = deque([n for n in nodes if indeg[n] == 0])
    layer = {n: 0 for n in q}
    topo_order = []

    while q:
        u = q.popleft()
        topo_order.append(u)
        for v in succ[u]:
            layer[v] = max(layer.get(v, 0), layer[u] + 1)
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)

    for n in nodes:
        layer.setdefault(n, 0)

    return layer, topo_order


def barycenter_order(nodes, neighbors, prev_order):
    def score(node):
        refs = [prev_order[x] for x in neighbors[node] if x in prev_order]
        if not refs:
            return (10**9, node)
        return (sum(refs) / len(refs), node)
    return sorted(nodes, key=score)


def make_circuit_layout(nodes, edges, layer_gap=3.0, row_gap=1.4, sweep_rounds=4):
    layer, _ = compute_topological_layers(nodes, edges)

    pred = {n: [] for n in nodes}
    succ = {n: [] for n in nodes}
    for src, dst, _ in edges:
        succ[src].append(dst)
        pred[dst].append(src)

    layers = defaultdict(list)
    for n in nodes:
        layers[layer[n]].append(n)

    max_layer = max(layers) if layers else 0
    order_in_layer = {lv: sorted(layers[lv]) for lv in range(max_layer + 1)}

    for _ in range(sweep_rounds):
        prev_order = {}
        for lv in range(max_layer + 1):
            for idx, n in enumerate(order_in_layer[lv]):
                prev_order[n] = idx
        for lv in range(1, max_layer + 1):
            order_in_layer[lv] = barycenter_order(order_in_layer[lv], pred, prev_order)

        next_order = {}
        for lv in range(max_layer + 1):
            for idx, n in enumerate(order_in_layer[lv]):
                next_order[n] = idx
        for lv in range(max_layer - 1, -1, -1):
            order_in_layer[lv] = barycenter_order(order_in_layer[lv], succ, next_order)

    pos = {}
    for lv in range(max_layer + 1):
        cur = order_in_layer[lv]
        k = len(cur)
        for idx, n in enumerate(cur):
            y = ((k - 1) / 2 - idx) * row_gap
            x = lv * layer_gap
            pos[n] = (x, y)

    return pos, layer


def draw_context_circuit_style(context, figsize=(16, 8), node_size=900):
    G = nx.DiGraph()
    for n in context["all_nodes"]:
        G.add_node(n, role=context["node_role"][n])
    for src, dst, is_inv in context["edges"]:
        G.add_edge(src, dst, is_inv=is_inv)

    pos, layer = make_circuit_layout(context["all_nodes"], context["edges"])

    color_map = {
        "core": "#d62728",
        "up": "#1f77b4",
        "down": "#2ca02c",
        "both": "#9467bd",
    }
    node_colors = [color_map[G.nodes[n]["role"]] for n in G.nodes]
    node_widths = [2.5 if G.nodes[n]["role"] == "core" else 1.2 for n in G.nodes]

    normal_edges = [(u, v) for u, v, d in G.edges(data=True) if d["is_inv"] == 0]
    inv_edges = [(u, v) for u, v, d in G.edges(data=True) if d["is_inv"] == 1]

    plt.figure(figsize=figsize)

    nx.draw_networkx_nodes(
        G,
        pos,
        node_color=node_colors,
        node_size=node_size,
        edgecolors="black",
        linewidths=node_widths,
    )
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edges(
        G, pos,
        edgelist=normal_edges,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.8,
    )
    nx.draw_networkx_edges(
        G, pos,
        edgelist=inv_edges,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=18,
        width=1.8,
        style="dashed",
        edge_color="orange",
    )

    max_layer = max(layer.values()) if layer else 0
    for lv in range(max_layer + 1):
        xs = [p[0] for n, p in pos.items() if layer[n] == lv]
        ys = [p[1] for n, p in pos.items() if layer[n] == lv]
        if xs and ys:
            x = xs[0]
            plt.plot([x, x], [min(ys) - 1.0, max(ys) + 1.0], color="#cccccc", linewidth=0.8, zorder=0)
            plt.text(x, max(ys) + 1.4, f"L{lv}", ha="center", va="bottom", fontsize=10)

    plt.title("Subgraph Context (Circuit-Style Left-to-Right Layout)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
with open(f"/home/zhoulingfeng/EDAProject/cutLiborigin/data/case_full_graphs/{dataset1}/{case1}/feat_graph.json", "r") as f:
    feat_graph_raw_1 = json.load(f)

context = build_context_from_all_core_nodes(
    subgraph_raw=data_1['cell_struct']['2345']['feats']['subgraph'],
    feat_graph_raw=feat_graph_raw_1,
    up_n=1,
    down_n=1,
)

In [ ]:
draw_context_circuit_style(context)

In [ ]:
feat_graph_1[13096]

In [2]:
import json

with open('/home/zhoulingfeng/EDAProject/cutLiborigin/log/cutlib_baseline_neighbor_analysis.json', 'r') as f:
    query_groups = json.load(f)

In [8]:
for query in query_groups['query_results']:
    case = query['query']['case']
    if 'chien' in case:
        print(case)

chien_search_p16_t16
chien_search_p16_t16
chien_search_p16_t16
chien_search_p32_t8
chien_search_p16_t16
chien_search_p16_t16


In [9]:
import scienceplots
